# Calse 3 - SQL con MySQL sobre Titanic (Data Cleaning)

Este notebook replica practicas de analisis preliminar, data wrangling y data cleaning, pero ejecutando consultas SQL sobre MySQL.

## Objetivo de la clase
Trabajar el flujo completo de limpieza y analisis preliminar sobre el dataset `titanic3.csv`, usando MySQL como motor de consulta y Jupyter como entorno de trabajo con Python.

Al finalizar, el estudiante podra:

- Conectar Jupyter con MySQL.
- Cargar un CSV a una tabla MySQL.
- Hacer analisis preliminar con SQL (dimension, tipos, nulos, distribuciones).
- Aplicar practicas de data wrangling con consultas SQL.
- Construir una vista de datos limpios para analisis posterior.

### Dataset de trabajo

- Archivo principal: `datasets/titanic3.csv`
- Tipo: censo de pasajeros del Titanic (no incluye tripulacion)
- Tamano: 1309 filas x 14 columnas

### De que trata este CSV
Cada fila representa un pasajero del Titanic y contiene variables demograficas, de viaje y de resultado (si sobrevivio o no). Es un dataset clasico para practicar:

- analisis exploratorio,
- limpieza de datos,
- tratamiento de valores nulos,
- ingenieria de variables,
- y modelado posterior (clasificacion binaria).

## 2) Diccionario de columnas (titanic3.csv)

| Columna | Tipo esperado | Descripcion |
|---|---|---|
| `pclass` | Entero (categorica ordinal) | Clase del pasajero (`1`=primera, `2`=segunda, `3`=tercera). Proxy de nivel socioeconomico. |
| `survived` | Entero binario | Supervivencia (`0`=no, `1`=si). |
| `name` | Texto | Nombre completo del pasajero. |
| `sex` | Texto | Sexo del pasajero (`male`, `female`). |
| `age` | Decimal | Edad en anios. Puede ser fraccional para menores de 1 anio. |
| `sibsp` | Entero | Numero de hermanos/as o esposo/a a bordo. |
| `parch` | Entero | Numero de padres/hijos a bordo. |
| `ticket` | Texto | Numero de ticket. |
| `fare` | Decimal | Tarifa pagada por el pasajero (libras historicas). |
| `cabin` | Texto | Cabina (muchos valores faltantes). |
| `embarked` | Texto | Puerto de embarque (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton). |
| `boat` | Texto | Bote salvavidas (si aplica). |
| `body` | Entero/Decimal | Numero de identificacion del cuerpo (si aplica). |
| `home.dest` | Texto | Origen o destino del pasajero. |

Nota tecnica para MySQL: en el notebook se renombra `home.dest` a `home_dest` para evitar problemas por el punto en el nombre de columna.

### Valores faltantes detectados

| Columna | Nulos |
|---|---:|
| `age` | 263 |
| `fare` | 1 |
| `cabin` | 1014 |
| `embarked` | 2 |
| `boat` | 823 |
| `body` | 1188 |
| `home.dest` | 564 |

Las demas columnas no presentan nulos en el CSV.


## 4) Levantar MySQL con Docker Compose

Crear archivo `docker-compose.yml` (en la raiz del proyecto):

```yaml
services:
  mysql:
    image: mysql:8.0
    container_name: mysql-titanic
    environment:
      MYSQL_ROOT_PASSWORD: root123
      MYSQL_DATABASE: titanic_ds
    ports:
      - "3307:3306"
    volumes:
      - mysql_titanic_data:/var/lib/mysql
    command: --default-authentication-plugin=mysql_native_password

volumes:
  mysql_titanic_data:
```

Levantar el servicio:

```bash
docker compose up -d
docker compose ps
```


## 0. Instalar Dependencias

In [ ]:
pip install pandas sqlalchemy pymysql jupyterlab

## 1) Dependencias y configuracion

- os → interacción con el sistema operativo.
- Path → manejo elegante de rutas de archivos.
- pandas → análisis y manipulación de datos.
- sqlalchemy → conexión y consultas a bases de datos.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

In [ ]:
# Ajusta estos valores si cambiaste tu docker-compose
MYSQL_USER = "root"
MYSQL_PASSWORD = "root123"
MYSQL_HOST = "localhost"
MYSQL_PORT = "3307"
MYSQL_DB = "titanic_ds"

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"
)

with engine.connect() as conn:
    print(conn.execute(text("SELECT 'Conexion OK' AS estado;")).fetchone())

## 2) Cargar CSV y subirlo a MySQL

In [ ]:
csv_path = Path('datasets/titanic3.csv')
df = pd.read_csv(csv_path)

# Normalizar nombres para SQL
df = df.rename(columns={'home.dest': 'home_dest'})

df.head()

In [ ]:
df.to_sql('titanic_passengers', con=engine, if_exists='replace', index=False, chunksize=500, method='multi')
print('Tabla cargada: titanic_passengers')

In [ ]:
pd.read_sql("SELECT COUNT(*) AS total_filas FROM titanic_passengers", engine)

## 3) Análisis preliminar
- Conteo total de filas.
- Conteo por `pclass`, `sex`, `survived`, `embarked`.
- Resumen numerico de `age`, `fare`, `sibsp`, `parch`.
- Deteccion de nulos por columna con `SUM(CASE WHEN ... THEN 1 END)`.

In [ ]:
q_schema = """
SELECT
  COLUMN_NAME AS columna,
  DATA_TYPE AS tipo,
  IS_NULLABLE AS permite_nulos
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = DATABASE()
  AND TABLE_NAME = 'titanic_passengers'
ORDER BY ORDINAL_POSITION;
"""
pd.read_sql(q_schema, engine)

In [ ]:
q_nulos = """
SELECT
  SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS age_nulls,
  SUM(CASE WHEN fare IS NULL THEN 1 ELSE 0 END) AS fare_nulls,
  SUM(CASE WHEN cabin IS NULL THEN 1 ELSE 0 END) AS cabin_nulls,
  SUM(CASE WHEN embarked IS NULL THEN 1 ELSE 0 END) AS embarked_nulls,
  SUM(CASE WHEN boat IS NULL THEN 1 ELSE 0 END) AS boat_nulls,
  SUM(CASE WHEN body IS NULL THEN 1 ELSE 0 END) AS body_nulls,
  SUM(CASE WHEN home_dest IS NULL THEN 1 ELSE 0 END) AS home_dest_nulls
FROM titanic_passengers;
"""
pd.read_sql(q_nulos, engine)

In [ ]:
q_distribucion = """
SELECT pclass, sex, survived, COUNT(*) AS total
FROM titanic_passengers
GROUP BY pclass, sex, survived
ORDER BY pclass, sex, survived;
"""
pd.read_sql(q_distribucion, engine)

Esa consulta construye un resumen estadístico básico de la tabla titanic_passengers, calculando en una sola fila el total de registros y, para las columnas age y fare, el número de valores no nulos (COUNT(columna)), el promedio (AVG redondeado a 4 decimales), el mínimo (MIN) y el máximo (MAX). En otras palabras, sirve para obtener rápidamente una visión general de cuántos datos válidos hay en esas variables y cuáles son sus principales medidas descriptivas, lo que resulta útil para evaluar la calidad y distribución inicial de la información antes de un análisis más profundo.

In [ ]:
q_resumen = """
SELECT
  COUNT(*) AS total_registros,
  COUNT(age) AS age_count,
  ROUND(AVG(age), 4) AS age_mean,
  MIN(age) AS age_min,
  MAX(age) AS age_max,
  COUNT(fare) AS fare_count,
  ROUND(AVG(fare), 4) AS fare_mean,
  MIN(fare) AS fare_min,
  MAX(fare) AS fare_max
FROM titanic_passengers;
"""
pd.read_sql(q_resumen, engine)

## 4) Data Wrangling con SQL
- Seleccion de columnas (subconjuntos).
- Filtros de filas por condiciones multiples.
- Ordenamiento y limites (`ORDER BY`, `LIMIT`).
- Creacion de tablas temporales o vistas para subconjuntos de interes.

El **Data Wrangling** es el proceso de preparar y transformar datos crudos para que puedan ser utilizados en análisis, visualización o modelado. Implica **limpiar** inconsistencias, manejar valores nulos, convertir formatos, integrar fuentes distintas y reorganizar la información para que sea más coherente y útil. En otras palabras, es como “domar” los datos: pasarlos de un estado desordenado y poco confiable a uno estructurado y listo para extraer conocimiento o alimentar modelos de inteligencia artificial y herramientas de análisis.

In [ ]:
q_subset = """
SELECT name, pclass, sex, age, fare, embarked
FROM titanic_passengers
LIMIT 20;
"""
pd.read_sql(q_subset, engine)

In [ ]:
q_filtros = """
SELECT name, pclass, sex, age, fare, survived
FROM titanic_passengers
WHERE pclass IN (1, 2)
  AND age IS NOT NULL
  AND age >= 18
  AND fare >= 30
ORDER BY fare DESC
LIMIT 30;
"""
pd.read_sql(q_filtros, engine)

In [ ]:
q_survival = """
SELECT
  pclass,
  sex,
  COUNT(*) AS total,
  ROUND(AVG(survived) * 100, 2) AS survival_rate_pct
FROM titanic_passengers
GROUP BY pclass, sex
ORDER BY pclass, sex;
"""
pd.read_sql(q_survival, engine)

## 5) Data Cleaning con SQL
- Imputacion de `age` con promedio por `pclass` y `sex` (fallback a promedio global).
- Imputacion de `embarked` con moda.
- Imputacion de `fare` con promedio por `pclass`.
- Variables derivadas:
  - `family_size = sibsp + parch + 1`
  - `is_alone` (1/0)
  - `age_group` (ninio, joven, adulto, adulto_mayor)
- Deteccion de outliers en `fare` usando regla IQR.

En este bloque creamos una vista limpia con imputaciones y variables derivadas.

Esta consulta crea o reemplaza una **vista llamada `titanic_clean`** que genera una versión depurada y enriquecida de la tabla `titanic_passengers`. Para ello, primero define varias subconsultas (`WITH`):  
- `age_stats` calcula el promedio de edad por clase (`pclass`) y sexo, mientras que `global_age` obtiene el promedio global de edad.  
- `fare_stats` calcula el promedio de tarifa por clase y `global_fare` el promedio global de tarifa.  
- `embarked_mode` identifica el puerto de embarque más frecuente (la moda).  

Luego, en la selección principal, se reemplazan los valores nulos de `age`, `fare` y `embarked` usando `COALESCE`: primero con el promedio por grupo, si no existe con el promedio global, y en el caso de `embarked` con la moda. Además, se agregan nuevas variables derivadas:  
- `family_size` calcula el tamaño de la familia sumando hermanos/esposos (`sibsp`), padres/hijos (`parch`) y el propio pasajero.  
- `is_alone` indica si el pasajero viajaba solo.  
- `age_group` clasifica la edad en categorías (`ninio`, `adolescente`, `adulto`, `adulto_mayor`).  

En resumen, esta vista estandariza valores faltantes y añade atributos útiles para análisis, convirtiéndose en una tabla limpia y lista para aplicar modelos o explorar patrones en los datos del Titanic.

In [ ]:
q_create_view = """
CREATE OR REPLACE VIEW titanic_clean AS
WITH age_stats AS (
  SELECT pclass, sex, AVG(age) AS age_avg
  FROM titanic_passengers
  WHERE age IS NOT NULL
  GROUP BY pclass, sex
),
global_age AS (
  SELECT AVG(age) AS age_avg_global
  FROM titanic_passengers
  WHERE age IS NOT NULL
),
fare_stats AS (
  SELECT pclass, AVG(fare) AS fare_avg
  FROM titanic_passengers
  WHERE fare IS NOT NULL
  GROUP BY pclass
),
global_fare AS (
  SELECT AVG(fare) AS fare_avg_global
  FROM titanic_passengers
  WHERE fare IS NOT NULL
),
embarked_mode AS (
  SELECT embarked
  FROM titanic_passengers
  WHERE embarked IS NOT NULL
  GROUP BY embarked
  ORDER BY COUNT(*) DESC
  LIMIT 1
)
SELECT
  t.pclass,
  t.survived,
  t.name,
  t.sex,
  COALESCE(t.age, a.age_avg, ga.age_avg_global) AS age_filled,
  t.sibsp,
  t.parch,
  t.ticket,
  COALESCE(t.fare, f.fare_avg, gf.fare_avg_global) AS fare_filled,
  t.cabin,
  COALESCE(t.embarked, em.embarked) AS embarked_filled,
  t.boat,
  t.body,
  t.home_dest,
  (t.sibsp + t.parch + 1) AS family_size,
  CASE WHEN (t.sibsp + t.parch) = 0 THEN 1 ELSE 0 END AS is_alone,
  CASE
    WHEN COALESCE(t.age, a.age_avg, ga.age_avg_global) < 13 THEN 'ninio'
    WHEN COALESCE(t.age, a.age_avg, ga.age_avg_global) < 18 THEN 'adolescente'
    WHEN COALESCE(t.age, a.age_avg, ga.age_avg_global) < 60 THEN 'adulto'
    ELSE 'adulto_mayor'
  END AS age_group
FROM titanic_passengers t
LEFT JOIN age_stats a
  ON t.pclass = a.pclass AND t.sex = a.sex
LEFT JOIN fare_stats f
  ON t.pclass = f.pclass
CROSS JOIN global_age ga
CROSS JOIN global_fare gf
CROSS JOIN embarked_mode em;
"""

with engine.begin() as conn:
    conn.execute(text(q_create_view))

print('Vista creada: titanic_clean')

In [ ]:
pd.read_sql("SELECT * FROM titanic_clean LIMIT 20", engine)

In [ ]:
q_total = """
SELECT COUNT(*) AS total FROM titanic_clean;
"""
pd.read_sql(q_total, engine)

Esta consulta construye un análisis para detectar **outliers en la columna `fare`** de la tabla `titanic_passengers` usando el método del rango intercuartílico (IQR).  

1. **CTE `ordered`**: ordena las tarifas (`fare`) y asigna un número de fila (`ROW_NUMBER`) a cada registro, además de calcular el total de filas (`COUNT(*) OVER ()`).  
2. **CTE `quartiles`**: a partir de esa numeración, extrae los valores de los cuartiles Q1 y Q3 usando expresiones condicionales (`CASE WHEN rn = ...`).  
3. **Selección principal**: se cruzan los pasajeros con los cuartiles calculados (`CROSS JOIN quartiles`).  
   - Se calcula el **IQR** como `q3 - q1`.  
   - Se filtran los registros cuyo `fare` está fuera del rango permitido:  
     - Menor que `Q1 - 1.5 * IQR`  
     - Mayor que `Q3 + 1.5 * IQR`  
   Estos son los valores considerados **outliers**.  
4. **Salida**: muestra el nombre del pasajero, su clase (`pclass`), la tarifa (`fare`), los cuartiles y el IQR, ordenando los resultados por tarifa descendente.  

👉 En resumen, la consulta implementa el criterio estadístico clásico para identificar valores atípicos en las tarifas del Titanic, permitiendo detectar pasajeros con boletos inusualmente baratos o caros respecto al resto.

In [ ]:
q_outliers_fare = """
WITH ordered AS (
  SELECT
    fare,
    ROW_NUMBER() OVER (ORDER BY fare) AS rn,
    COUNT(*) OVER () AS cnt
  FROM titanic_passengers
  WHERE fare IS NOT NULL
),
quartiles AS (
  SELECT
    MAX(CASE WHEN rn = FLOOR((cnt + 3) / 4) THEN fare END) AS q1,
    MAX(CASE WHEN rn = FLOOR((3 * cnt + 1) / 4) THEN fare END) AS q3
  FROM ordered
)
SELECT
  t.name,
  t.pclass,
  t.fare,
  q.q1,
  q.q3,
  (q.q3 - q.q1) AS iqr
FROM titanic_passengers t
CROSS JOIN quartiles q
WHERE t.fare IS NOT NULL
  AND (
    t.fare < (q.q1 - 1.5 * (q.q3 - q.q1))
    OR t.fare > (q.q3 + 1.5 * (q.q3 - q.q1))
  )
ORDER BY t.fare DESC;
"""
pd.read_sql(q_outliers_fare, engine).head(20)

## 6) Exportar resultados

In [ ]:
df_clean = pd.read_sql("SELECT * FROM titanic_clean", engine)
df_survival = pd.read_sql(q_survival, engine)

df_clean.to_csv('titanic_clean.csv', index=False, encoding='utf-8')
df_survival.to_csv('titanic_survival_by_group.csv', index=False, encoding='utf-8')

print('Archivos exportados: titanic_clean.csv, titanic_survival_by_group.csv')

---